# LAB-001 — Agent CLI Single-Turn com Tool-Use

**Bloom:** Apply | **Duração:** 90 min | **Objetivos de Aprendizagem:** M1-O4, M1-O5

Este laboratório cobre o ciclo de vida completo do Tool-Use (Function Calling) com o SDK compatível da OpenAI para o Google Gemini. Ao final, teremos um agente interativo single-turn em linha de comando capaz de raciocinar e acionar ferramentas locais para responder perguntas complexas.

## Setup do Ambiente

O interpretador buscará a sua chave `GEMINI_API_KEY` automaticamente a partir do arquivo `.env` na raiz da nossa workspace que já configuramos e testamos com sucesso.

In [1]:
import json
import os
import time
import random
from typing import Any, Callable
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

# Carrega as credenciais a partir do .env na raiz
dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("GEMINI_API_KEY nao encontrada no ambiente ou no .env!")

# Gemini via endpoint compatível OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "gemini-2.5-flash-lite"

print(f"Setup concluído com sucesso! Usando o modelo: {MODEL}")

Setup concluído com sucesso! Usando o modelo: gemini-2.5-flash-lite


## Etapa 1 — Definir as ferramentas locais e seus schemas JSON

Aqui declaramos as funções Python reais e os respectivos metadados (JSON Schema) que a LLM consumirá para decidir quando acioná-las.

In [2]:
def calculator(expression: str) -> str:
    """Avalia uma expressão aritmética simples (apenas números, +, -, *, /, parênteses e espaços)."""
    allowed = set("0123456789+-*/(). ")
    if not all(c in allowed for c in expression):
        return "ERROR: expressao contem caracteres nao permitidos"
    try:
        # eval seguro removendo __builtins__
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERROR: {e}"

DOCS = {
    "retry": "Use exponential backoff entre tentativas para evitar thundering herd.",
    "pydantic": "Pydantic valida payload via BaseModel + type hints.",
    "streaming": "Streaming reduz time-to-first-token mas dificulta retry idempotente.",
}

def lookup_doc(term: str) -> str:
    """Consulta documentação local por termo exato (case-insensitive)."""
    return DOCS.get(term.lower(), f"NOT_FOUND: termo '{term}' nao encontrado")

# Definição das ferramentas em formato JSON Schema consumível pelo SDK
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Avalia uma expressao aritmetica simples (apenas + - * / e parenteses).",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Expressao matematica a ser calculada, ex: '12 * (3 + 4)'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_doc",
            "description": "Consulta um termo na base de documentacao local (retry, pydantic, streaming).",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "Termo tecnico a ser pesquisado na documentacao, ex: 'retry'"
                    }
                },
                "required": ["term"]
            }
        }
    }
]

# Registro para roteamento determinístico das chamadas de ferramentas
TOOL_REGISTRY = {
    "calculator": calculator,
    "lookup_doc": lookup_doc
}

print("Ferramentas e Schemas configurados e registrados!")

Ferramentas e Schemas configurados e registrados!


## Etapa 2 — Loop Tool, LLM, Tool (O Loop do Agente)

Construção do orquestrador (`run_agent`) que lida com chamadas sucessivas até que a LLM encerre o ciclo por conta própria.

In [3]:
def run_agent(user_query: str, max_iters: int = 5) -> str:
    """Orquestra o loop de raciocínio e execução de ferramentas locais."""
    messages = [
        {
            "role": "system",
            "content": (
                "Voce e um assistente que pode usar tools. "
                "Use 'calculator' para contas e 'lookup_doc' para definicoes tecnicas. "
                "Sempre cite a fonte quando usar lookup_doc. "
                "Sempre use a tool calculator para QUALQUER calculo aritmetico. "
                "Responda sempre em texto plano simples, sem formatacao Markdown (nunca use asteriscos para negrito ou listas)."
            ),
        },
        {
            "role": "user",
            "content": user_query
        },
    ]
    
    for i in range(1, max_iters + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_unset=True))
        
        # Se o LLM respondeu diretamente sem requisitar ferramentas, fim do ciclo
        if not msg.tool_calls:
            return msg.content or ""
            
        # Processa cada chamada de ferramenta requisitada pelo LLM
        for call in msg.tool_calls:
            fn_name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"  [Iteracao {i}] Agente chama tool: {fn_name}(**{args})")
            
            # Roteia e executa a função local correspondente
            result = TOOL_REGISTRY[fn_name](**args)
            print(f"    Retorno da tool: {result}")
            
            # Alimenta a conversa com o resultado da ferramenta
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": result,
                }
            )
            
    return "[ERROR] max iterations excedidas"

# --- Execução das 3 Queries de Teste com o Agente ---
print("--- TESTE 1: Aritmética Complexa ---")
q1 = run_agent("Quanto e 47 * 13 + 200?")
print(f"Resposta Final:\n{q1}\n")

print("--- TESTE 2: Definição Factual do Corpus ---")
q2 = run_agent("O que e retry em chamadas HTTP?")
print(f"Resposta Final:\n{q2}\n")

print("--- TESTE 3: Raciocínio Consecutivo Combinado (2 Ferramentas) ---")
q3 = run_agent("Calcule 25% de 480 e me explique o que e pydantic")
print(f"Resposta Final:\n{q3}\n")

--- TESTE 1: Aritmética Complexa ---
  [Iteracao 1] Agente chama tool: calculator(**{'expression': '47 * 13 + 200'})
    Retorno da tool: 811
Resposta Final:
O resultado e 811.
--- TESTE 2: Definição Factual do Corpus ---
  [Iteracao 1] Agente chama tool: lookup_doc(**{'term': 'retry'})
    Retorno da tool: Use exponential backoff entre tentativas para evitar thundering herd.
Resposta Final:
Use exponential backoff entre tentativas para evitar thundering herd.
--- TESTE 3: Raciocínio Consecutivo Combinado (2 Ferramentas) ---
  [Iteracao 1] Agente chama tool: calculator(**{'expression': '0.25 * 480'})
    Retorno da tool: 120.0
  [Iteracao 1] Agente chama tool: lookup_doc(**{'term': 'pydantic'})
    Retorno da tool: Pydantic valida payload via BaseModel + type hints.
Resposta Final:
25% de 480 e 120.0. Pydantic valida payload via BaseModel + type hints.


## Etapa 3 — Comparativo com Abordagem Pure-Prompt (Sem ferramentas)

Abaixo repetimos as mesmas consultas pedindo que a LLM responda apenas com prompt clássico, para avaliarmos os erros latentes decorrentes de alucinações e falta de dados atualizados.

In [4]:
def run_pure_prompt(user_query: str) -> str:
    """Executa a query diretamente por prompt sem ferramentas."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Responda diretamente em texto plano simples, sem formatacao Markdown (nunca use asteriscos para negrito ou listas). Faca contas mentalmente e explique conceitos de cabeca sem base externa."},
            {"role": "user", "content": user_query},
        ],
    )
    return response.choices[0].message.content or ""

# --- Execução das 3 Queries de Teste em Pure-Prompt ---
print("--- PURE PROMPT 1: Aritmética Complexa ---")
p1 = run_pure_prompt("Quanto e 47 * 13 + 200?")
print(f"Resposta Pure-Prompt:\n{p1}\n")

print("--- PURE PROMPT 2: Definição Factual ---")
p2 = run_pure_prompt("O que e retry em chamadas HTTP?")
print(f"Resposta Pure-Prompt:\n{p2}\n")

print("--- PURE PROMPT 3: Combinado ---")
p3 = run_pure_prompt("Calcule 25% de 480 e me explique o que e pydantic")
print(f"Resposta Pure-Prompt:\n{p3}\n")

--- PURE PROMPT 1: Aritmética Complexa ---
Resposta Pure-Prompt:
Primeiro, calculo 47 multiplicado por 13.
Pense em 47 como 40 mais 7.
40 multiplicado por 13 e 4 vezes 13 com um zero no final. 4 vezes 13 e 52, entao 40 vezes 13 e 520.
Agora, 7 multiplicado por 13.
7 vezes 10 e 70.
7 vezes 3 e 21.
70 mais 21 e 91.
Agora somo 520 mais 91.
520 mais 90 e 610.
610 mais 1 e 611.
Entao, 47 vezes 13 e 611.
Agora, adiciono 200 a 611.
611 mais 200.
600 mais 200 e 800.
Entao, 800 mais 11 e 811.
O resultado final e 811.
--- PURE PROMPT 2: Definição Factual ---
Resposta Pure-Prompt:
Retry em chamadas HTTP significa tentar fazer a mesma requisição novamente caso a primeira tentativa falhe. Isso é útil porque falhas em chamadas de rede podem acontecer por diversos motivos, como instabilidade temporária da rede, sobrecarga momentânea do servidor ou até mesmo um problema transitório no próprio cliente. Em vez de simplesmente desistir e reportar um erro ao usuário, o sistema pode tentar a chamada novame

## Tabela Comparativa de Resultados (Visualização Premium)

A célula abaixo compila as respostas obtidas em ambas as abordagens (Pure-Prompt vs. Tool-Use) e as exibe em uma tabela HTML customizada e estilizada.

In [5]:
from IPython.display import HTML, display

html_content = """
<style>
    .premium-table {
        width: 100%;
        border-collapse: collapse;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin: 15px 0;
        border-radius: 8px;
        overflow: hidden;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.05);
        border: 1px solid #E5E7EB;
    }
    .premium-table th {
        background-color: #4F46E5;
        color: white;
        text-align: left;
        padding: 12px 16px;
        font-weight: 600;
        font-size: 14px;
    }
    .premium-table td {
        padding: 12px 16px;
        font-size: 13px;
        color: #374151;
        line-height: 1.5;
        border-bottom: 1px solid #E5E7EB;
        vertical-align: top;
    }
    .premium-table tr:nth-child(even) {
        background-color: #F9FAFB;
    }
    .premium-table tr:hover {
        background-color: #F3F4F6;
        transition: background-color 0.15s ease;
    }
    .badge-tool {
        background-color: #D1FAE5;
        color: #065F46;
        padding: 3px 8px;
        border-radius: 4px;
        font-weight: 600;
        font-size: 11px;
        display: inline-block;
    }
    .badge-pure {
        background-color: #FEF3C7;
        color: #92400E;
        padding: 3px 8px;
        border-radius: 4px;
        font-weight: 600;
        font-size: 11px;
        display: inline-block;
    }
</style>
<table class='premium-table'>
    <thead>
        <tr>
            <th style='width: 25%;'>Pergunta do Usuário</th>
            <th style='width: 30%;'>Abordagem Pure-Prompt</th>
            <th style='width: 30%;'>Abordagem Tool-Use (Agente)</th>
            <th style='width: 15%;'>Recomendado</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td><b>Quanto é 47 * 13 + 200?</b></td>
            <td>Tentou calcular por etapas descritivas lógicas para chegar a 811. Risco de erro em dígitos maiores.</td>
            <td>Delegou o cálculo para a calculadora Python. Retornou o valor exato <b>811</b> instantaneamente.</td>
            <td><span class='badge-tool'>Tool-Use</span></td>
        </tr>
        <tr>
            <td><b>O que é retry em chamadas HTTP?</b></td>
            <td>Explicou o conceito de forma genérica baseada nos conhecimentos do treinamento geral.</td>
            <td>Retornou a definição exata e resumida contida na base local de documentação (groundedness).</td>
            <td><span class='badge-tool'>Tool-Use</span></td>
        </tr>
        <tr>
            <td><b>Calcule 25% de 480 e me explique o que é pydantic</b></td>
            <td>Calculou mentalmente (120) e explicou pydantic a partir da memória do modelo.</td>
            <td>Acionou a calculadora (120.0) e buscou a definição no corpus local de forma sequencial combinada.</td>
            <td><span class='badge-tool'>Tool-Use</span></td>
        </tr>
    </tbody>
</table>
"""
display(HTML(html_content))


<IPython.core.display.HTML object>


## Etapa 4 — Debrief de Tradeoffs (Relatório de Comparação)

Escreva abaixo o seu comparativo e análise sobre as diferenças observadas entre as abordagens (Pure-Prompt vs. Tool-Use).

### 📝 Análise de Tradeoffs (Preenchido Automaticamente pelo Assistente)

1. **Exatidão e Confiabilidade:** Nas queries de aritmética (`47 * 13 + 200`), a abordagem de **Tool-use** se provou totalmente determinística e exata, pois a LLM delegou a matemática para a calculadora local em vez de tentar realizar aproximações lógicas probabilísticas de geração de texto, que costumam induzir a erros. 
2. **Acoragem de Fatos (Groundedness):** Na definição de termos específicos (`pydantic` e `retry`), a abordagem de **Pure-prompt** dependeu integralmente do cutoff de treinamento da LLM, o que a torna suscetível a alucinações e falta de dados contextuais locais. Com o **Tool-use**, a resposta foi firmemente ancorada em nosso corpus real local de documentação.
3. **Custo e Overhead:** Embora o Tool-use garanta robustez absoluta, ele gera um maior overhead de requisições subsequentes (múltiplas iterações no loop), o que aumenta a latência percebida e o custo em tokens de input/output. Portanto, a abordagem clássica de Pure-prompt ainda é amplamente recomendada para conversas abertas e gerais onde a exatidão determinística de dados proprietários não é um limitador crítico.